In [2]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns
import os

# ---------------------------------------------------------------------
# CUSTOM LAYER FOR QUANTUM MODELS
# ---------------------------------------------------------------------
# Replace this with your actual custom layer implementation or import.
class QuantumInspiredLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, inputs):
        return inputs

# ---------------------------------------------------------------------
# CLASS NAMES AND TEST DIRECTORY
# ---------------------------------------------------------------------
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']

TEST_DIR = r"C:\Users\hassa\OneDrive\Desktop\QML\Brain_Tumor_data\Testing"

# ---------------------------------------------------------------------
# LOAD TEST DATA AS NUMPY ARRAY
# ---------------------------------------------------------------------
IMG_SIZE = (64, 64)   # MODIFY if your input size differs

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMG_SIZE,
    shuffle=False,
    batch_size=32
)

# Extract images and labels into arrays
X_test = np.concatenate([x for x, _ in test_ds], axis=0)
y_test = np.concatenate([y for _, y in test_ds], axis=0)

# Normalize if needed
X_test = X_test / 255.0

# ---------------------------------------------------------------------
# MODEL PATHS (Raw string r"" for Windows paths)
# ---------------------------------------------------------------------
model_paths = {
    "ML-CNN": {
        "ADAM":    r"C:\Users\hassa\OneDrive\Desktop\QML\ADAM\models\classical_cnn_complete.keras",
        "ADAGRAD": r"C:\Users\hassa\OneDrive\Desktop\QML\ADAGRAD\cnn\adagrad_models\classical_cnn_Adagrad_complete.keras",
        "SGD":     r"C:\Users\hassa\OneDrive\Desktop\QML\SDG\sdg-models\classical_cnn_SGD_complete.keras"
    },
    "Quantum-CNN": {
        "ADAM":    r"C:\Users\hassa\OneDrive\Desktop\QML\ADAM\models\quantum_cnn_brain_tumor_model.keras",
        "ADAGRAD": r"C:\Users\hassa\OneDrive\Desktop\QML\ADAGRAD\qcnn\models\quantum_cnn_brain_tumor_model.keras",
        "SGD":     r"C:\Users\hassa\OneDrive\Desktop\QML\SDG\sdg-models\quantum_cnn_brain_tumor_model.keras"
    },
    "Federated-CNN": {
        "ADAM":    r"C:\Users\hassa\OneDrive\Desktop\QML\ADAM\models\federated_model_round_5.keras",
        "ADAGRAD": r"C:\Users\hassa\OneDrive\Desktop\QML\ADAGRAD\flcnn\sdg-models\federated_model_round_5.keras",
        "SGD":     r"C:\Users\hassa\OneDrive\Desktop\QML\SDG\sdg-models\federated_model_round_5.keras"
    },
    "Federated-Quantum-CNN": {
        "ADAM":    r"C:\Users\hassa\OneDrive\Desktop\QML\ADAM\models\federated_quantum_model_round_5.keras",
        "ADAGRAD": r"C:\Users\hassa\OneDrive\Desktop\QML\ADAGRAD\flqcnn\flq-models\federated_quantum_model_round_5.keras",
        "SGD":     r"C:\Users\hassa\OneDrive\Desktop\QML\SDG\flqcnn\flq-models\federated_quantum_model_round_5.keras"
    },
}

model_types = ["ML-CNN", "Quantum-CNN", "Federated-CNN", "Federated-Quantum-CNN"]
optimizers = ["ADAM", "ADAGRAD", "SGD"]

# ---------------------------------------------------------------------
# CONFUSION MATRIX PLOT FUNCTION
# ---------------------------------------------------------------------
def plot_cm(ax, y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        cbar=False,
        ax=ax
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

# ---------------------------------------------------------------------
# GENERATE 4 × 3 CONFUSION MATRIX GRID
# ---------------------------------------------------------------------
fig, axes = plt.subplots(
    nrows=4, ncols=3,
    figsize=(21, 28),    # Suitable for research paper PDF/PNG
    dpi=300
)

for i, model_type in enumerate(model_types):
    for j, optimizer in enumerate(optimizers):

        model_path = model_paths[model_type][optimizer]
        ax = axes[i, j]

        print(f"\nLoading: {model_type} - {optimizer}")

        # Quantum models need custom_objects
        if "Quantum" in model_type:
            model = load_model(model_path, custom_objects={"QuantumInspiredLayer": QuantumInspiredLayer})
        else:
            model = load_model(model_path)

        # Predict
        y_pred_probs = model.predict(X_test, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)

        title = f"{model_type} | {optimizer}"
        plot_cm(ax, y_test, y_pred, title)

plt.tight_layout()
output_file = "confusion_matrix_12_models.png"
plt.savefig(output_file, dpi=300, bbox_inches="tight")
plt.close()

print(f"\nSaved high-resolution figure as: {output_file}")


ImportError: Traceback (most recent call last):
  File "c:\Users\hassa\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 73, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [ ]:
ModelType,Optimizer,TimeSec (s),TrainAcc,ValAcc,TrainLoss,ValLoss,LR
FEDERATED CNN,ADAM,7.7337 ± 0.0925,0.7983 ± 0.0671,0.9463 ± 0.0236,0.5447 ± 0.1841,0.1920 ± 0.0295,0.0010 ± 0.0000
FEDERATED CNN,Adagrad,7.7354 ± 0.0923,0.7117 ± 0.1208,0.4342 ± 0.0974,0.8061 ± 0.3316,1.8067 ± 0.2777,0.0010 ± 0.0000
FEDERATED CNN,SDG,7.7908 ± 0.0562,0.7340 ± 0.1112,0.3635 ± 0.1201,0.7262 ± 0.3084,1.9424 ± 0.5657,0.0010 ± 0.0000
FEDERATED QUANTUM CNN,ADAM,7.8038 ± 0.2171,0.5798 ± 0.0913,0.6854 ± 0.0361,1.0138 ± 0.1591,0.7552 ± 0.0850,0.0010 ± 0.0000
FEDERATED QUANTUM CNN,Adagrad,7.8957 ± 0.0806,0.5588 ± 0.1512,0.3641 ± 0.0632,1.1254 ± 0.1475,1.3089 ± 0.0622,0.0010 ± 0.0000
FEDERATED QUANTUM CNN,SDG,8.1645 ± 0.0470,0.6076 ± 0.1850,1.3469 ± 0.0749*,0.9071 ± 0.2979,0.4422 ± 0.0741,0.0010 ± 0.0000
MLCNN,ADAM,4.9063 ± 0.0575,0.9741 ± 0.0063,0.9666 ± 0.0048,0.0775 ± 0.0196,0.1269 ± 0.0211,0.0010 ± 0.0000
MLCNN,Adagrad,4.9063 ± 0.0575,0.8374 ± 0.0191,0.8050 ± 0.0384,0.4512 ± 0.0471,0.5963 ± 0.1050,0.0010 ± 0.0000
MLCNN,SDG,4.9063 ± 0.0575,0.9741 ± 0.0063,0.9666 ± 0.0048,0.0775 ± 0.0196,0.1269 ± 0.0211,0.0010 ± 0.0000
QUANTUM CNN,ADAM,4.8542 ± 0.2157,0.6852 ± 0.0169,0.7304 ± 0.0082,0.7509 ± 0.0376,0.6680 ± 0.0335,0.0007 ± 0.0003
QUANTUM CNN,Adagrad,4.8542 ± 0.2157,0.7290 ± 0.0849,0.7437 ± 0.0707,0.9074 ± 0.0766,0.8895 ± 0.0725,0.0010 ± 0.0000
QUANTUM CNN,SDG,4.6547 ± 0.1508,0.8862 ± 0.0145,0.8904 ± 0.0108,0.3719 ± 0.0311,0.3780 ± 0.0298,0.0008 ± 0.0003

In [4]:
import pandas as pd

def generate_research_summary(input_file, output_file):
    # 1. Load the dataset
    df = pd.read_csv(input_file)

    # 2. Preprocessing: Handle Federated Learning Clients
    # Group by Model, Optimizer, Round, and Epoch to average metrics across all clients
    # This creates a single representative row for each epoch of a global model
    group_cols = ['ModelType', 'Optimizer', 'Round_Fold', 'Epoch']
    df_epoch = df.groupby(group_cols).mean(numeric_only=True).reset_index()

    # 3. Calculate Average Training Time per Epoch (for each Run/Fold)
    # Instead of taking the time of just the "best" epoch (which is noisy), 
    # we take the average time of all epochs in that specific run.
    avg_time_per_run = df_epoch.groupby(['ModelType', 'Optimizer', 'Round_Fold'])['TimeSec'].mean().reset_index()
    avg_time_per_run = avg_time_per_run.rename(columns={'TimeSec': 'AvgTimeSec'})

    # 4. Selection: Identify the Best Epoch for each Run
    # We select the epoch with the highest Validation Accuracy (ValAcc)
    def get_best_epoch(x):
        return x.sort_values('ValAcc', ascending=False).iloc[0]

    best_epochs = df_epoch.groupby(['ModelType', 'Optimizer', 'Round_Fold']).apply(get_best_epoch).reset_index(drop=True)

    # 5. Merge the Average Time back into the best epoch data
    # We drop the specific 'TimeSec' of the best epoch and replace it with the run's average
    best_epochs = pd.merge(best_epochs.drop(columns=['TimeSec']), 
                           avg_time_per_run, 
                           on=['ModelType', 'Optimizer', 'Round_Fold'])

    # 6. Aggregation: Calculate Mean and Std Dev across Folds
    # We group by Model and Optimizer (aggregating the 5 folds)
    metrics = ['AvgTimeSec', 'TrainAcc', 'ValAcc', 'TrainLoss', 'ValLoss', 'LR']
    summary_stats = best_epochs.groupby(['ModelType', 'Optimizer'])[metrics].agg(['mean', 'std'])

    # 7. Formatting: Create the final "Mean ± Std" strings
    final_table = pd.DataFrame()
    
    # Map internal column names to display names if needed
    column_mapping = {
        'AvgTimeSec': 'Time (s)',
        'TrainAcc': 'Train Acc',
        'ValAcc': 'Val Acc',
        'TrainLoss': 'Train Loss',
        'ValLoss': 'Val Loss',
        'LR': 'Learning Rate'
    }

    for metric in metrics:
        display_name = column_mapping.get(metric, metric)
        final_table[display_name] = summary_stats[metric].apply(
            lambda x: f"{x['mean']:.4f} ± {x['std']:.4f}", axis=1
        )

    # Reset index to make ModelType and Optimizer regular columns
    final_table = final_table.reset_index()

    # 8. Save and Display
    print("Generated Summary Table:")
    print(final_table.to_markdown(index=False))
    
    final_table.to_csv(output_file, index=False)
    print(f"\nSuccessfully saved summary to {output_file}")

if __name__ == "__main__":
    generate_research_summary(r'C:\Users\hassa\OneDrive\Desktop\QML\FINAL_RESEARCH_DATA.csv', 'summary_table_research.csv')

KeyError: 'ValAcc'